# Task 3: Data Cleaning and Text Preprocessing
## Activity 1: Load and Review the Dataset

In [36]:
import pandas as pd
import numpy as np
import re
import ast
from pathlib import Path

# Load the original dataset
data_path = Path("../data/raw/ted_talks_en.csv")
df = pd.read_csv(data_path)

print("Original dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Duplicate rows:", df.duplicated().sum())

display(df.head())

Original dataset loaded successfully!
Rows: 4005
Columns: 19
Duplicate rows: 0


,talk_id,title,speaker_1,all_speakers,occupations,about_speakers,views,recorded_date,published_date,event,native_lang,available_lang,comments,duration,topics,related_talks,url,description,transcript
0,1,Averting the climate crisis,Al Gore,{0: 'Al Gore'},{0: ['climate advocate']},{0: 'Nobel Laureate Al Gore focused the world’...,3523392,2006-02-25,2006-06-27,TED2006,en,"['ar', 'bg', 'cs', 'de', 'el', 'en', 'es', 'fa...",272.0,977,"['alternative energy', 'cars', 'climate change...","{243: 'New thinking on the climate crisis', 54...",https://www.ted.com/talks/al_gore_averting_the...,With the same humor and humanity he exuded in ...,"Thank you so much, Chris. And it's truly a gre..."
1,92,The best stats you've ever seen,Hans Rosling,{0: 'Hans Rosling'},{0: ['global health expert; data visionary']},"{0: 'In Hans Rosling’s hands, data sings. Glob...",14501685,2006-02-22,2006-06-27,TED2006,en,"['ar', 'az', 'bg', 'bn', 'bs', 'cs', 'da', 'de...",628.0,1190,"['Africa', 'Asia', 'Google', 'demo', 'economic...","{2056: ""Own your body's data"", 2296: 'A visual...",https://www.ted.com/talks/hans_rosling_the_bes...,You've never seen data presented like this. Wi...,"About 10 years ago, I took on the task to teac..."
2,7,Simplicity sells,David Pogue,{0: 'David Pogue'},{0: ['technology columnist']},{0: 'David Pogue is the personal technology co...,1920832,2006-02-24,2006-06-27,TED2006,en,"['ar', 'bg', 'de', 'el', 'en', 'es', 'fa', 'fr...",124.0,1286,"['computers', 'entertainment', 'interface desi...","{1725: '10 top time-saving tech tips', 2274: '...",https://www.ted.com/talks/david_pogue_simplici...,New York Times columnist David Pogue takes aim...,"(Music: ""The Sound of Silence,"" Simon & Garfun..."
3,53,Greening the ghetto,Majora Carter,{0: 'Majora Carter'},{0: ['activist for environmental justice']},{0: 'Majora Carter redefined the field of envi...,2664069,2006-02-26,2006-06-27,TED2006,en,"['ar', 'bg', 'bn', 'ca', 'cs', 'de', 'en', 'es...",219.0,1116,"['MacArthur grant', 'activism', 'business', 'c...",{1041: '3 stories of local eco-entrepreneurshi...,https://www.ted.com/talks/majora_carter_greeni...,"In an emotionally charged talk, MacArthur-winn...",If you're here today — and I'm very happy that...
4,66,Do schools kill creativity?,Sir Ken Robinson,{0: 'Sir Ken Robinson'},"{0: ['author', 'educator']}","{0: ""Creativity expert Sir Ken Robinson challe...",65051954,2006-02-25,2006-06-27,TED2006,en,"['af', 'ar', 'az', 'be', 'bg', 'bn', 'ca', 'cs...",4931.0,1164,"['children', 'creativity', 'culture', 'dance',...","{865: 'Bring on the learning revolution!', 173...",https://www.ted.com/talks/sir_ken_robinson_do_...,Sir Ken Robinson makes an entertaining and pro...,Good morning. How are you? (Audience) Good. It...


In [37]:
# Review missing values in the original dataset
missing_report = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": (
        df.isnull().sum() / len(df) * 100
    ).round(2)
})

display(missing_report[missing_report["Missing Count"] > 0])

,Missing Count,Missing Percentage
all_speakers,4,0.10
occupations,522,13.03
about_speakers,503,12.56
recorded_date,1,0.02
comments,655,16.35


## Activity 2: Prepare a Working Copy

In [38]:
# Work on a separate copy so the original data remains unchanged
df_clean = df.copy()

print("Original rows:", len(df))
print("Working-copy rows:", len(df_clean))
print("Original dataset remains unchanged.")

Original rows: 4005
Working-copy rows: 4005
Original dataset remains unchanged.


In [39]:
# Identify missing and empty values in important columns
important_columns = [
    "title", "topics", "description", "transcript", "url"
]

for col in important_columns:
    if col in df_clean.columns:
        text = df_clean[col].astype("string").str.strip()
        empty_count = (text.isna() | text.eq("")).sum()

        print(
            f"{col}: missing={df_clean[col].isna().sum()}, "
            f"missing_or_empty={empty_count}"
        )

title: missing=0, missing_or_empty=0
topics: missing=0, missing_or_empty=0
description: missing=0, missing_or_empty=0
transcript: missing=0, missing_or_empty=0
url: missing=0, missing_or_empty=0


### Missing-Value Handling Decisions

1. Title: Keep available titles. If a title is missing, retain the record and handle the missing title as empty text in the recommendation features.
2. Transcript: Do not remove talks with missing transcripts if other useful fields are available.
3. Description and topics: Preserve available information and use empty text when these fields are missing during text combination.
4. Comments, occupations, and speaker descriptions: Retain missing values because these fields are optional supporting information.
5. URL: Retain the record and review missing URLs separately if any are found.
6. No unnecessary records will be deleted during missing-value handling.

## Activity 3: Duplicate Record Handling

In [40]:
# Record duplicate counts before cleaning
duplicate_rows_before = df_clean.duplicated().sum()

duplicate_ids_before = (
    df_clean["talk_id"].duplicated().sum()
    if "talk_id" in df_clean.columns else 0
)

duplicate_urls_before = (
    df_clean["url"].dropna().duplicated().sum()
    if "url" in df_clean.columns else 0
)

print("Duplicate rows before cleaning:", duplicate_rows_before)
print("Repeated talk IDs before cleaning:", duplicate_ids_before)
print("Repeated URLs before cleaning:", duplicate_urls_before)

Duplicate rows before cleaning: 0
Repeated talk IDs before cleaning: 0
Repeated URLs before cleaning: 0


In [41]:
# remove exact duplicate rows and record the count

rows_before_dedup = len(df_clean)

df_clean = df_clean.drop_duplicates().copy()

exact_duplicates_removed = rows_before_dedup - len(df_clean)

print("Rows before duplicate removal:", rows_before_dedup)
print("Exact duplicate rows removed:", exact_duplicates_removed)
print("Rows after duplicate removal:", len(df_clean))

Rows before duplicate removal: 4005
Exact duplicate rows removed: 0
Rows after duplicate removal: 4005


In [42]:
# Inspect repeated identifiers and URLs
if "talk_id" in df_clean.columns:
    repeated_ids = df_clean[
        df_clean["talk_id"].duplicated(keep=False)
    ].sort_values("talk_id")

    print("Rows with repeated talk IDs:", len(repeated_ids))
    display(repeated_ids.head(10))

if "url" in df_clean.columns:
    valid_urls = df_clean["url"].notna() & (
        df_clean["url"].astype("string").str.strip() != ""
    )

    repeated_urls = df_clean.loc[valid_urls].loc[
        df_clean.loc[valid_urls, "url"].duplicated(keep=False)
    ].sort_values("url")

    print("Rows with repeated URLs:", len(repeated_urls))
    display(repeated_urls.head(10))

Rows with repeated talk IDs: 0


,talk_id,title,speaker_1,all_speakers,occupations,about_speakers,views,recorded_date,published_date,event,native_lang,available_lang,comments,duration,topics,related_talks,url,description,transcript


Rows with repeated URLs: 0


,talk_id,title,speaker_1,all_speakers,occupations,about_speakers,views,recorded_date,published_date,event,native_lang,available_lang,comments,duration,topics,related_talks,url,description,transcript


## Activity 4: Text Cleaning and Normalization

In [43]:
def clean_text(value):
    """Normalize text while preserving meaningful words."""
    if pd.isna(value):
        return ""

    text = str(value).lower()

    # Remove URLs from text content if present
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # Replace punctuation with spaces while retaining
    # apostrophes and hyphens within words
    text = re.sub(r"[^\w\s'-]", " ", text)

    # Remove underscores and normalize whitespace
    text = text.replace("_", " ")
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [44]:
# Clean the title, description, and transcript

text_columns = ["title", "description", "transcript"]

for col in text_columns:
    if col in df_clean.columns:
        df_clean[col + "_clean"] = df_clean[col].apply(clean_text)

print("Text cleaning completed.")

Text cleaning completed.


In [45]:
# Clean and normalize topics

def parse_topics(value):
    if pd.isna(value):
        return []

    if isinstance(value, list):
        items = value
    else:
        value = str(value).strip()

        try:
            parsed = ast.literal_eval(value)
            items = parsed if isinstance(parsed, list) else [value]
        except (ValueError, SyntaxError):
            items = [value]

    return [
        clean_text(item)
        for item in items
        if clean_text(item)
    ]

if "topics" in df_clean.columns:
    df_clean["topics_clean"] = df_clean["topics"].apply(parse_topics)

print("Topic cleaning completed.")
display(df_clean[["topics", "topics_clean"]].head())

Topic cleaning completed.


,topics,topics_clean
0,"['alternative energy', 'cars', 'climate change...","[alternative energy, cars, climate change, cul..."
1,"['Africa', 'Asia', 'Google', 'demo', 'economic...","[africa, asia, google, demo, economics, global..."
2,"['computers', 'entertainment', 'interface desi...","[computers, entertainment, interface design, m..."
3,"['MacArthur grant', 'activism', 'business', 'c...","[macarthur grant, activism, business, cities, ..."
4,"['children', 'creativity', 'culture', 'dance',...","[children, creativity, culture, dance, educati..."


In [46]:
# Compare original and cleaned text

comparison_columns = [
    col for col in [
        "title", "title_clean",
        "description", "description_clean",
        "transcript", "transcript_clean"
    ]
    if col in df_clean.columns
]

display(df_clean[comparison_columns].head(5))

,title,title_clean,description,description_clean,transcript,transcript_clean
0,Averting the climate crisis,averting the climate crisis,With the same humor and humanity he exuded in ...,with the same humor and humanity he exuded in ...,"Thank you so much, Chris. And it's truly a gre...",thank you so much chris and it's truly a great...
1,The best stats you've ever seen,the best stats you've ever seen,You've never seen data presented like this. Wi...,you've never seen data presented like this wit...,"About 10 years ago, I took on the task to teac...",about 10 years ago i took on the task to teach...
2,Simplicity sells,simplicity sells,New York Times columnist David Pogue takes aim...,new york times columnist david pogue takes aim...,"(Music: ""The Sound of Silence,"" Simon & Garfun...",music the sound of silence simon garfunkel hel...
3,Greening the ghetto,greening the ghetto,"In an emotionally charged talk, MacArthur-winn...",in an emotionally charged talk macarthur-winni...,If you're here today — and I'm very happy that...,if you're here today and i'm very happy that y...
4,Do schools kill creativity?,do schools kill creativity,Sir Ken Robinson makes an entertaining and pro...,sir ken robinson makes an entertaining and pro...,Good morning. How are you? (Audience) Good. It...,good morning how are you audience good it's be...


## Activity 5: Create the Combined Text Field

In [47]:
def combine_text(row):
    parts = []

    for col in [
        "title_clean",
        "topics_clean",
        "description_clean",
        "transcript_clean"
    ]:
        if col not in row.index:
            continue

        value = row[col]

        if isinstance(value, list):
            value = " ".join(value)

        if isinstance(value, str) and value.strip():
            parts.append(value.strip())

    return " ".join(parts)

df_clean["combined_text"] = df_clean.apply(combine_text, axis=1)

print("Combined text created successfully.")
display(
    df_clean[["title", "topics", "combined_text"]].head(5)
)

Combined text created successfully.


,title,topics,combined_text
0,Averting the climate crisis,"['alternative energy', 'cars', 'climate change...",averting the climate crisis alternative energy...
1,The best stats you've ever seen,"['Africa', 'Asia', 'Google', 'demo', 'economic...",the best stats you've ever seen africa asia go...
2,Simplicity sells,"['computers', 'entertainment', 'interface desi...",simplicity sells computers entertainment inter...
3,Greening the ghetto,"['MacArthur grant', 'activism', 'business', 'c...",greening the ghetto macarthur grant activism b...
4,Do schools kill creativity?,"['children', 'creativity', 'culture', 'dance',...",do schools kill creativity children creativity...


In [48]:
# Check that it is populated

print("Empty combined_text entries:",
      df_clean["combined_text"].str.strip().eq("").sum())

print("Missing combined_text entries:",
      df_clean["combined_text"].isna().sum())

Empty combined_text entries: 0
Missing combined_text entries: 0


In [49]:

# Compare rows before and after cleaning

# Convert list values into tuples for duplicate checking
def make_hashable(df):
    df_copy = df.copy()
    for col in df_copy.columns:
        df_copy[col] = df_copy[col].apply(
            lambda x: tuple(x) if isinstance(x, list) else x
        )
    return df_copy

df_check = make_hashable(df)
df_clean_check = make_hashable(df_clean)

comparison_report = pd.DataFrame({
    "Metric": [
        "Rows",
        "Exact duplicate rows",
        "Repeated talk IDs"
    ],
    "Before Cleaning": [
        len(df),
        int(df_check.duplicated().sum()),
        int(df["talk_id"].duplicated().sum())
        if "talk_id" in df.columns else "N/A"
    ],
    "After Cleaning": [
        len(df_clean),
        int(df_clean_check.duplicated().sum()),
        int(df_clean["talk_id"].duplicated().sum())
        if "talk_id" in df_clean.columns else "N/A"
    ]
})

display(comparison_report)


,Metric,Before Cleaning,After Cleaning
0,Rows,4005,4005
1,Exact duplicate rows,0,0
2,Repeated talk IDs,0,0


## Activity 6: Save the Cleaned Dataset

In [50]:
# Create the processed-data folder if it does not exist
output_path = Path("../data/processed/ted_talks_cleaned.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

# Save the cleaned working copy
df_clean.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully!")
print("File path:", output_path.resolve())
print("Final dataset shape:", df_clean.shape)

Cleaned dataset saved successfully!
File path: C:\Users\THARANI NAVEENA\Desktop\TED-Talks\data\processed\ted_talks_cleaned.csv
Final dataset shape: (4005, 24)


In [51]:
# Read the saved dataset to verify it
saved_df = pd.read_csv(output_path)

print("Saved dataset rows:", saved_df.shape[0])
print("Saved dataset columns:", saved_df.shape[1])
print("combined_text exists:", "combined_text" in saved_df.columns)

display(saved_df[["title", "combined_text"]].head())

Saved dataset rows: 4005
Saved dataset columns: 24
combined_text exists: True


,title,combined_text
0,Averting the climate crisis,averting the climate crisis alternative energy...
1,The best stats you've ever seen,the best stats you've ever seen africa asia go...
2,Simplicity sells,simplicity sells computers entertainment inter...
3,Greening the ghetto,greening the ghetto macarthur grant activism b...
4,Do schools kill creativity?,do schools kill creativity children creativity...


## Task 3: Data Cleaning and Text Preprocessing Report

The original TED Talks dataset was loaded from the raw data folder, and a separate working copy was created to preserve the original dataset.

Missing values in important text fields were reviewed. Missing transcripts will not automatically cause talks to be removed because titles, topics, or descriptions may still provide useful information. Optional fields such as comments, occupations, and speaker descriptions are retained.

Exact duplicate rows, repeated talk IDs, and repeated URLs were checked. Only verified exact duplicate rows were removed. The before-and-after comparison records the number of rows and duplicates.

Text preprocessing included lowercase conversion, punctuation normalization, removal of unnecessary whitespace, and consistent handling of missing text. Original text columns were preserved, and separate cleaned columns were created for comparison.

A new `combined_text` column was created using the available title, topics, description, and transcript. The processed dataset was saved to `data/processed/ted_talks_cleaned.csv` without overwriting the original CSV.

**Limitations:** Lowercase conversion and punctuation removal can change some text details. Short transcripts may contain applause, music, or stage directions. These cases require review, and cleaning should avoid removing useful information. The combined text is prepared for the next stage of text vectorization and similarity-based recommendations.